# BioBERT + LoRA: Radyoloji Raporu Siniflandirma (18 sinif)

> ⚠️ **Klinik kullanim icin degildir.** Egitim/portfolyo projesidir.
> Veri: *Chameleon Radiology Reports* (satvikt04, Hugging Face) — **sentetik** raporlar,
> yalnizca arastirma/egitim amacli, yeniden dagitilamaz.

**Baslamadan once:** Menuden `Runtime > Change runtime type > T4 GPU` sec.

## 1) GPU kontrolu
Beklenen: `Tesla T4` yazan bir tablo.

In [ ]:
!nvidia-smi

## 2) Kurulum (surumler sabit)
torch'u **kurmuyoruz**: Colab'in kendi GPU'lu torch'u kullanilacak.
Kirmizi "dependency conflict" uyarilari cikabilir (Colab'in baska paketleriyle ilgili), gormezden gel.
"Restart session" dugmesi cikarsa tikla ve bu hucreden **sonraki** hucreden devam et.

In [ ]:
!pip install -q transformers==5.18.0 peft==0.21.1 datasets==5.0.1 accelerate==1.15.0

In [ ]:
import torch, transformers, peft, datasets, accelerate, sklearn
for m in (torch, transformers, peft, datasets, accelerate, sklearn):
    print(f"{m.__name__:<13} {m.__version__}")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "YOK!")

## 3) `prepare.py` dosyasini yukle
Bilgisayarindaki `prepare.py` dosyasini sec. Egitimde ve serviste **ayni** metin hazirlama kullanilsin diye
fonksiyonu kopyalamiyoruz, dosyanin kendisini yukluyoruz.

In [ ]:
import os
if not os.path.exists("prepare.py"):
    from google.colab import files
    files.upload()  # prepare.py'yi sec
from prepare import prepare_report
print("prepare.py hazir")

## 4) Veriyi yukle ve bol (%80 egitim / %10 dogrulama / %10 test)
`stratify` = her bolmede sinif oranlari ayni kalsin (Healthy %50, digerleri ~%3).

In [ ]:
from datasets import load_dataset

DATASET_ID = "satvikt04/Chameleon-Radiology-Reports"
DATASET_REVISION = "ebf03d97a699b6ad61fc20c9a93f08b54da13f06"
MODEL_ID = "dmis-lab/biobert-v1.1"
MODEL_REVISION = "551ca18efd7f052c8dfa0b01c94c2a8e68bc5488"
SEED = 42

ds = load_dataset(DATASET_ID, revision=DATASET_REVISION, split="train")
ds = ds.class_encode_column("Pathology")  # metin etiket -> sayi (alfabetik sira)
labels = ds.features["Pathology"].names
id2label = dict(enumerate(labels))
label2id = {l: i for i, l in id2label.items()}

split = ds.train_test_split(test_size=0.2, stratify_by_column="Pathology", seed=SEED)
val_test = split["test"].train_test_split(test_size=0.5, stratify_by_column="Pathology", seed=SEED)
train_ds, val_ds, test_ds = split["train"], val_test["train"], val_test["test"]

print(f"egitim={len(train_ds)}  dogrulama={len(val_ds)}  test={len(test_ds)}")
print(f"{len(labels)} sinif:", labels)

## 5) Metni hazirla ve token'lara cevir

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)

def preprocess(batch):
    texts = [prepare_report(t) for t in batch["Report"]]
    enc = tokenizer(texts, truncation=True, max_length=512)
    enc["labels"] = batch["Pathology"]
    return enc

cols = train_ds.column_names
train_tok = train_ds.map(preprocess, batched=True, remove_columns=cols)
val_tok = val_ds.map(preprocess, batched=True, remove_columns=cols)
test_tok = test_ds.map(preprocess, batched=True, remove_columns=cols)
print(train_tok)

## 6) BioBERT + LoRA
- Model yuklenirken *"Some weights ... newly initialized: classifier"* uyarisi **normal**:
  18 cikisli siniflandirma katmani yeni ekleniyor.
- `r=8`: adapter'in boyutu (not defterinin sayfa sayisi gibi). `target_modules`: LoRA'yi dikkat
  katmanlarinin `query` ve `value` parcalarina ekliyoruz.

In [ ]:
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, TaskType, get_peft_model

base = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION,
    num_labels=len(labels), id2label=id2label, label2id=label2id,
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,   # siniflandirma: classifier katmani da tamamen egitilir
    r=8, lora_alpha=16, lora_dropout=0.1,
    target_modules=["query", "value"],
)
model = get_peft_model(base, lora_config)
model.print_trainable_parameters()

## 7) Egitim (~15-25 dk, T4)
- **epoch** = egitim verisinin tamamini bir kez gormek. 3 epoch yapiyoruz.
- Her epoch sonunda dogrulama verisinde **accuracy** ve **macro-F1** olculur;
  en iyi macro-F1'li epoch saklanir.
- LoRA'da ogrenme hizi (`learning_rate`) tam fine-tuning'den yuksek secilir (2e-4).

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments

def compute_metrics(eval_pred):
    logits, y = eval_pred
    pred = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(y, pred), "macro_f1": f1_score(y, pred, average="macro")}

args = TrainingArguments(
    output_dir="checkpoints",
    num_train_epochs=3,
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=50,
    fp16=True,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)
trainer.train()

## 8) Test seti: modelin hic gormedigi veri
Sinif bazinda **precision** (model "X" dediginde ne siklikla hakli), **recall** (gercek X'lerin
ne kadarini yakaladi) ve **F1** (ikisinin dengesi).

In [ ]:
from sklearn.metrics import classification_report

out = trainer.predict(test_tok)
y_true, y_pred = out.label_ids, out.predictions.argmax(-1)
print(out.metrics)
print(classification_report(y_true, y_pred, target_names=labels, digits=3))

## 9) Kendi yazdigimiz metinlerle deneme
Veri setinden rapor **alintilamiyoruz** (kullanim sartlari). Asagidaki cumleler bizim yazdigimiz ornekler.

In [ ]:
model.eval()
examples = [
    "IMPRESSION: No acute cardiopulmonary abnormality. FINDINGS: Lungs are clear. No pleural effusion.",
    "IMPRESSION: Moderate right pleural effusion with adjacent atelectasis.",
    "IMPRESSION: Filling defect in the right lower lobe pulmonary artery consistent with acute pulmonary embolism.",
]
for text in examples:
    enc = tokenizer(prepare_report(text), truncation=True, max_length=512, return_tensors="pt").to(model.device)
    with torch.no_grad():
        probs = model(**enc).logits.softmax(-1)[0]
    top = probs.argmax().item()
    print(f"{probs[top]:.3f}  {id2label[top]:<35} <- {text[:60]}...")

## 10) Adapter'i kaydet ve indir
Kaydedilen klasorde sadece **adapter + classifier** agirliklari var (birkac MB), BioBERT'in kendisi yok.
Colab oturumu kapaninca dosyalar silinir; zip'i bilgisayarina indir (yedek). Hub'a yukleme Adim 5'te.

In [ ]:
SAVE_DIR = "biobert-lora-radiology"
trainer.model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
!ls -lh {SAVE_DIR}
!zip -qr {SAVE_DIR}.zip {SAVE_DIR}

from google.colab import files
files.download(f"{SAVE_DIR}.zip")